# 04 — Function Scope and LEGB

## Functions & Scope

This notebook focuses on **where variables are accessible** and how Python **resolves variable names**.

By the end, you will understand:

- local and global scope
- enclosing scope in nested functions
- the LEGB rule
- reading and modifying global variables
- the `global` and `nonlocal` keywords
- variable shadowing
- common scope-related errors


## 1. What Is Scope?

**Scope determines where a variable can be accessed in a Python program.**

For example, a variable created inside a function normally belongs to that function's local scope.


In [ ]:
def greet():
    message = "Hello"
    print(message)

greet()


`message` exists inside `greet()`.

Trying to access it outside the function causes a `NameError` because the name is not available in the current scope.

```python
print(message)
```

The variable is local to `greet()`.


In [ ]:
def greet():
    message = "Hello"
    print(message)

greet()

# print(message)  # NameError


## 2. Local Scope

Variables created inside a function are **local variables**.

They can be used within that function.


In [ ]:
def calculate():
    x = 10
    y = 20
    print(x + y)

calculate()


Here, `x` and `y` belong to the local scope of `calculate()`.

### Each function call has its own local scope

Calling a function creates its local variables for that call. A later call gets its own local state.


In [ ]:
def show_value():
    value = 10
    print(value)

show_value()
show_value()


The local variable `value` belongs to each function call. One function call does not create a single shared local variable for all calls.


## 3. Global Scope

A variable defined outside functions belongs to the **global scope** of the module.


In [ ]:
x = 100

def show():
    print(x)

show()


`x` is defined outside `show()`, so the function can find it in the global scope.

A function can **read** an accessible global variable without declaring it with `global`.


## 4. Local vs Global Variables

A local variable can have the same name as a global variable.


In [ ]:
x = 100

def test():
    x = 50
    print(x)

test()
print(x)


Output:

```text
50
100
```

The `x` inside `test()` is a different local variable from the global `x`.

This is an example of **variable shadowing**: the local name hides the global name within that local scope.


## 5. LEGB Rule

The **LEGB rule** describes the order Python uses when resolving a variable name:

```text
L → Local
E → Enclosing
G → Global
B → Built-in
```

Python searches for a name in this order:

1. **Local** — the current function
2. **Enclosing** — an outer function, when nested functions are involved
3. **Global** — the module-level scope
4. **Built-in** — Python's built-in names


### Local

Python checks the current function's local scope first.


In [ ]:
x = "global"

def test():
    x = "local"
    print(x)

test()


The local `x` is found first, so `"local"` is printed.


### Enclosing

The enclosing scope is relevant when one function is nested inside another.


In [ ]:
def outer():
    x = "outer"

    def inner():
        print(x)

    inner()

outer()


`inner()` does not have a local `x`, so Python looks in the enclosing function `outer()` and finds `"outer"`.


### Global

If Python cannot find the name locally or in an enclosing function, it checks the global scope.


In [ ]:
x = "global"

def test():
    print(x)

test()


There is no local or enclosing `x`, so Python finds the global `x`.


### Built-in

Finally, Python can find names in the built-in namespace.

For example, `len` is a built-in function.


In [ ]:
print(len("Python"))


Python finds `len` in the built-in namespace when it is not defined in a nearer scope.

### Be careful with built-in names

You can create a variable named `print`, but doing so can hide the built-in `print()` function in that scope.


In [ ]:
print("Before shadowing")

print = "Hello"
print(print)

# The second print(...) call attempts to use the string stored in
# print, so it raises a TypeError.


This is why it is generally a bad idea to use names such as `print`, `len`, `sum`, or `list` for your own variables.

The important LEGB idea is:

```text
Local → Enclosing → Global → Built-in
```


## 6. Accessing Global Variables Inside Functions

Reading a global variable inside a function is allowed.


In [ ]:
count = 10

def show_count():
    print(count)

show_count()


The function does not create a local `count` merely by reading the global variable.

However, **modifying** a global variable is different.


## 7. Modifying Global Variables with `global`

Consider this example:

```python
count = 10

def update():
    count = count + 1
```

This produces an `UnboundLocalError`.

Why?

Because the assignment to `count` makes Python treat `count` as a **local variable** throughout `update()`. The right-hand side then tries to read that local variable before it has a value.


In [ ]:
count = 10

def update():
    # count = count + 1
    # Uncommenting the line above causes UnboundLocalError.
    pass

update()


To explicitly tell Python that `count` refers to the global variable, use `global`.


In [ ]:
count = 10

def update():
    global count
    count = count + 1

update()

print(count)


Output:

```text
11
```

The `global` statement tells Python that assignments to `count` inside the function should refer to the global variable instead of creating a local `count`.


## 8. The `global` Keyword

The `global` keyword is used when a function needs to **assign to a variable in the global scope**.

```python
count = 10

def update():
    global count
    count += 1
```

Without `global`, an assignment to `count` inside the function makes `count` local to that function.

### Important distinction

Reading:

```python
print(count)
```

can use an accessible global variable.

Assigning:

```python
count = count + 1
```

requires `global count` if the intention is to modify the global variable.


## 9. Scope in Nested Functions

Functions can be defined inside other functions.

In this example, `message` belongs to the outer function.


In [ ]:
def outer():
    message = "Hello"

    def inner():
        print(message)

    inner()

outer()


`message` is not local to `inner()`, but it is available in the **enclosing scope**.

This is the `E` in LEGB.

The detailed behavior of nested functions and closures is covered in the next dedicated notebook.


## 10. The `nonlocal` Keyword

Consider this example:

```python
def outer():
    count = 0

    def inner():
        count = 1
        print(count)

    inner()
    print(count)
```

The `count` inside `inner()` is a **new local variable**. It does not modify `outer()`'s `count`.


In [ ]:
def outer():
    count = 0

    def inner():
        count = 1
        print(count)

    inner()
    print(count)

outer()


To modify the variable from the enclosing function scope, use `nonlocal`.


In [ ]:
def outer():
    count = 0

    def inner():
        nonlocal count
        count += 1

    inner()
    print(count)

outer()


Output:

```text
1
```

`nonlocal` tells Python that `count` should refer to the variable in the nearest enclosing function scope rather than creating a new local variable.


## 11. `global` vs `nonlocal`

The two keywords refer to different scopes:

| Keyword | Refers to |
|---|---|
| `global` | Global scope |
| `nonlocal` | Enclosing function scope |

A combined example:


In [ ]:
x = 10

def outer():
    y = 20

    def inner():
        global x
        nonlocal y

        x += 1
        y += 1

    inner()
    print(y)

outer()
print(x)


Here:

- `global x` makes `x` refer to the module-level variable.
- `nonlocal y` makes `y` refer to the variable in `outer()`.
- `x += 1` changes the global `x`.
- `y += 1` changes the enclosing `y`.


## 12. Variable Shadowing

**Variable shadowing** occurs when a nearer scope defines a variable with the same name as a variable in an outer scope.

Consider three levels:


In [ ]:
x = "global"

def outer():
    x = "outer"

    def inner():
        x = "inner"
        print(x)

    inner()
    print(x)

outer()
print(x)


Output:

```text
inner
outer
global
```

Each `x` belongs to a different scope.

The example visualizes LEGB:

```text
inner x   → Local
outer x   → Enclosing
global x  → Global
```

Because `inner()` has its own local `x`, Python stops searching at the Local level.


## 13. Common Scope Errors

### `NameError`

A `NameError` occurs when Python cannot find a requested name in an accessible scope.

For example:


In [ ]:
def test():
    # x does not exist in Local, Enclosing, Global, or Built-in scope.
    # print(x)  # NameError
    pass

test()


### `UnboundLocalError`

A common cause is reading a name before assigning to it inside the same function.


In [ ]:
x = 10

def test():
    # Because x is assigned below, Python treats x as local
    # throughout this function.
    # print(x)
    # x = 20
    pass

test()


If the commented lines are uncommented, Python raises `UnboundLocalError`.

The important point is that Python determines the local binding because of the assignment:

```python
x = 20
```

So this:

```python
print(x)
x = 20
```

does not first use the global `x`.


## 14. Scope Resolution Examples

A useful way to understand LEGB is to start with an inner function and progressively remove its local variable.


### Example 1 — Local wins


In [ ]:
x = "Global"

def outer():
    x = "Enclosing"

    def inner():
        x = "Local"
        print(x)

    inner()

outer()


The output is:

```text
Local
```

Python finds `x` immediately in the local scope of `inner()`.


### Example 2 — Remove the local variable


In [ ]:
x = "Global"

def outer():
    x = "Enclosing"

    def inner():
        print(x)

    inner()

outer()


Now there is no local `x` in `inner()`, so Python finds the enclosing `x` from `outer()`.


### Example 3 — Remove the enclosing variable


In [ ]:
x = "Global"

def outer():

    def inner():
        print(x)

    inner()

outer()


Now there is no local or enclosing `x`, so Python finds the global `x`.


### Example 4 — Built-in scope


In [ ]:
def example():
    print(len("Python"))

example()


There is no local, enclosing, or global `len` in this example, so Python resolves `len` from the built-in namespace.

This gives the complete progression:

```text
Local
  ↓
Enclosing
  ↓
Global
  ↓
Built-in
```


## 15. Summary

### Scope

Scope determines where a variable can be accessed.

### Local scope

Variables created inside a function normally belong to that function's local scope.

### Global scope

Variables defined outside functions belong to the global scope.

### LEGB

Python resolves names in this order:

```text
L → Local
E → Enclosing
G → Global
B → Built-in
```

### `global`

Use `global` when a function needs to assign to a variable in the global scope.

```python
global count
```

### `nonlocal`

Use `nonlocal` when a nested function needs to assign to a variable in an enclosing function scope.

```python
nonlocal count
```

### Shadowing

A variable in a nearer scope can hide a variable with the same name in an outer scope.

### Final mental model

```text
Local
  ↓
Enclosing
  ↓
Global
  ↓
Built-in

global   → modify global variable
nonlocal → modify enclosing variable
```

The next notebook, `05_Nested_Functions_and_Closures.ipynb`, will build on nested-function scope and explain closures in depth.
